# Yelp Text Representation Pipeline

This notebook prepares leakage-safe textual representations for the
KGRec-inspired recommendation experiment.

For the controlled personalised experiment, business review-text
representations are derived only from training interactions. Review text
associated with validation and test interactions is excluded from feature
construction to prevent target leakage.

The broader Yelp review corpus is retained separately for general discovery
and later evidence-based explanation tasks.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Adjust only if your existing project paths differ
processed_data_dir = Path(
    "processed_data/new_orleans_subset"
)

image_pipeline_dir = Path(
    "processed_data/new_orleans_image_pipeline"
)

clip_output_dir = (
    image_pipeline_dir
    / "clip_vit_b32_embeddings"
)


# --------------------------------------------------
# Frozen interaction data
# --------------------------------------------------

train_interactions = pd.read_parquet(
    processed_data_dir
    / "new_orleans_positive_train.parquet"
)

validation_interactions = pd.read_parquet(
    processed_data_dir
    / "new_orleans_positive_validation.parquet"
)

test_interactions = pd.read_parquet(
    processed_data_dir
    / "new_orleans_positive_test.parquet"
)


# --------------------------------------------------
# Full New Orleans review corpus
# --------------------------------------------------

catalogue_reviews = pd.read_parquet(
    processed_data_dir
    / "new_orleans_food_hospitality_reviews_raw.parquet"
)


# --------------------------------------------------
# Personalisation businesses
# --------------------------------------------------

personalisation_businesses = pd.read_parquet(
    processed_data_dir
    / "new_orleans_personalisation_businesses.parquet"
)


# --------------------------------------------------
# Already-frozen visual availability information
# --------------------------------------------------

visual_feature_index = pd.read_parquet(
    clip_output_dir
    / "new_orleans_personalisation_visual_feature_index.parquet"
)


print("Train interactions:", f"{len(train_interactions):,}")
print("Validation interactions:", f"{len(validation_interactions):,}")
print("Test interactions:", f"{len(test_interactions):,}")

print(
    "Full review corpus:",
    f"{len(catalogue_reviews):,}"
)

print(
    "Personalisation businesses:",
    f"{personalisation_businesses['business_id'].nunique():,}"
)

print(
    "Visual feature index:",
    f"{len(visual_feature_index):,}"
)

Train interactions: 122,233
Validation interactions: 14,991
Test interactions: 14,991
Full review corpus: 559,117
Personalisation businesses: 2,516
Visual feature index: 2,516


## Stage E1 — Audit the review-text fields

Before joining anything, let's inspect what we actually have.

In [2]:
print(
    "Review columns:"
)

print(
    catalogue_reviews.columns.tolist()
)


review_text_audit = pd.Series({
    "Review rows":
        len(catalogue_reviews),

    "Unique review IDs":
        catalogue_reviews[
            "review_id"
        ].nunique(),

    "Missing review IDs":
        catalogue_reviews[
            "review_id"
        ].isna().sum(),

    "Missing business IDs":
        catalogue_reviews[
            "business_id"
        ].isna().sum(),

    "Missing user IDs":
        catalogue_reviews[
            "user_id"
        ].isna().sum(),

    "Missing text":
        catalogue_reviews[
            "text"
        ].isna().sum(),

    "Blank text":
        catalogue_reviews[
            "text"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum(),

    "Duplicate review IDs":
        catalogue_reviews[
            "review_id"
        ].duplicated().sum()
})

review_text_audit

Review columns:
['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date', 'interaction_sentiment', 'is_positive', 'is_negative', 'is_neutral']


Review rows             559117
Unique review IDs       559117
Missing review IDs           0
Missing business IDs         0
Missing user IDs             0
Missing text                 0
Blank text                   0
Duplicate review IDs         0
dtype: int64

In [3]:
training_review_text = (
    train_interactions[
        [
            "review_id",
            "user_id",
            "business_id",
            "stars",
            "date"
        ]
    ]
    .merge(
        catalogue_reviews[
            [
                "review_id",
                "text"
            ]
        ],
        on="review_id",
        how="left",
        validate="one_to_one"
    )
)


print(
    "Training interactions:",
    f"{len(train_interactions):,}"
)

print(
    "Matched training reviews:",
    f"{len(training_review_text):,}"
)

print(
    "Missing training review text:",
    training_review_text[
        "text"
    ].isna().sum()
)

Training interactions: 122,233
Matched training reviews: 122,233
Missing training review text: 0


In [4]:
training_review_ids = set(
    training_review_text[
        "review_id"
    ]
)

validation_review_ids = set(
    validation_interactions[
        "review_id"
    ]
)

test_review_ids = set(
    test_interactions[
        "review_id"
    ]
)


text_leakage_check = pd.Series({
    "Training text rows":
        len(
            training_review_text
        ),

    "Unique training review IDs":
        training_review_text[
            "review_id"
        ].nunique(),

    "Missing training text":
        training_review_text[
            "text"
        ].isna().sum(),

    "Blank training text":
        training_review_text[
            "text"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum(),

    "Validation review IDs in training text":
        len(
            training_review_ids
            & validation_review_ids
        ),

    "Test review IDs in training text":
        len(
            training_review_ids
            & test_review_ids
        ),

    "Training businesses represented":
        training_review_text[
            "business_id"
        ].nunique(),

    "Expected personalisation businesses":
        personalisation_businesses[
            "business_id"
        ].nunique()
})

text_leakage_check

Training text rows                        122233
Unique training review IDs                122233
Missing training text                          0
Blank training text                            0
Validation review IDs in training text         0
Test review IDs in training text               0
Training businesses represented             2516
Expected personalisation businesses         2516
dtype: int64

## Stage E2.1 — Audit Training Review Coverage and Text Length

The leakage-safe training review corpus is analysed before text embedding.
Review volume and textual length are examined at both review and business
levels to determine an appropriate representation and aggregation strategy.

The audit is used to avoid arbitrary review caps or document concatenation
that could disproportionately affect businesses with different amounts of
textual evidence.

In [5]:
training_review_text = (
    training_review_text
    .copy()
)

training_review_text[
    "text_clean"
] = (
    training_review_text[
        "text"
    ]
    .astype(str)
    .str.strip()
)

training_review_text[
    "word_count"
] = (
    training_review_text[
        "text_clean"
    ]
    .str.split()
    .str.len()
)

training_review_text[
    "character_count"
] = (
    training_review_text[
        "text_clean"
    ]
    .str.len()
)


review_length_summary = (
    training_review_text[
        [
            "word_count",
            "character_count"
        ]
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

review_length_summary

,word_count,character_count
count,122233.000000,122233.000000
mean,112.687433,615.535952
std,94.682179,518.077916
min,1.000000,1.000000
1%,15.000000,87.000000
5%,22.000000,124.000000
25%,48.000000,264.000000
50%,85.000000,466.000000
75%,147.000000,799.000000
90%,229.000000,1244.000000


In [6]:
business_text_summary = (
    training_review_text
    .groupby(
        "business_id"
    )
    .agg(
        training_review_count=(
            "review_id",
            "count"
        ),

        training_reviewer_count=(
            "user_id",
            "nunique"
        ),

        total_words=(
            "word_count",
            "sum"
        ),

        mean_review_words=(
            "word_count",
            "mean"
        ),

        median_review_words=(
            "word_count",
            "median"
        ),

        maximum_review_words=(
            "word_count",
            "max"
        )
    )
    .reset_index()
)


print(
    "Businesses represented:",
    len(
        business_text_summary
    )
)

print(
    "Unique businesses:",
    business_text_summary[
        "business_id"
    ].nunique()
)

Businesses represented: 2516
Unique businesses: 2516


In [7]:
business_review_count_summary = (
    business_text_summary[
        "training_review_count"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

business_review_count_summary

count    2516.000000
mean       48.582273
std        88.810075
min         1.000000
1%          3.000000
5%          4.000000
25%         8.000000
50%        19.000000
75%        51.000000
90%       110.000000
95%       173.000000
99%       477.050000
max      1271.000000
Name: training_review_count, dtype: float64

In [8]:
review_volume_groups = pd.cut(
    business_text_summary[
        "training_review_count"
    ],
    bins=[
        0,
        2,
        5,
        10,
        20,
        50,
        np.inf
    ],
    labels=[
        "1–2",
        "3–5",
        "6–10",
        "11–20",
        "21–50",
        ">50"
    ]
)


review_volume_distribution = (
    review_volume_groups
    .value_counts()
    .sort_index()
    .rename(
        "businesses"
    )
    .to_frame()
)

review_volume_distribution[
    "percentage"
] = (
    review_volume_distribution[
        "businesses"
    ]
    / len(
        business_text_summary
    )
    * 100
).round(2)

review_volume_distribution

,businesses,percentage
training_review_count,,
1–2,24,0.95
3–5,281,11.17
6–10,485,19.28
11–20,512,20.35
21–50,578,22.97
>50,636,25.28


In [9]:
duplicate_text_audit = pd.Series({
    "Training review rows":
        len(
            training_review_text
        ),

    "Exact duplicate text rows":
        training_review_text[
            "text_clean"
        ].duplicated().sum(),

    "Duplicate review text within same business":
        training_review_text
        .duplicated(
            subset=[
                "business_id",
                "text_clean"
            ]
        )
        .sum(),

    "Unique review texts":
        training_review_text[
            "text_clean"
        ].nunique()
})

duplicate_text_audit

Training review rows                          122233
Exact duplicate text rows                         37
Duplicate review text within same business         0
Unique review texts                           122196
dtype: int64

In [10]:
business_text_source_check = pd.Series({
    "Training review rows":
        len(
            training_review_text
        ),

    "Businesses represented":
        business_text_summary[
            "business_id"
        ].nunique(),

    "Total reviews represented in business summary":
        business_text_summary[
            "training_review_count"
        ].sum(),

    "Review totals exactly match":
        (
            business_text_summary[
                "training_review_count"
            ].sum()
            ==
            len(
                training_review_text
            )
        ),

    "Minimum training reviews for a business":
        business_text_summary[
            "training_review_count"
        ].min(),

    "Maximum training reviews for a business":
        business_text_summary[
            "training_review_count"
        ].max(),

    "Businesses with zero training reviews":
        (
            business_text_summary[
                "training_review_count"
            ]
            == 0
        ).sum()
})

business_text_source_check

Training review rows                             122233
Businesses represented                             2516
Total reviews represented in business summary    122233
Review totals exactly match                        True
Minimum training reviews for a business               1
Maximum training reviews for a business            1271
Businesses with zero training reviews                 0
dtype: object

## Stage E2.2 — Text Encoder Selection and Token-Length Audit

BAAI/bge-small-en-v1.5 is used as the pretrained textual encoder for the
personalised recommendation experiment.

The model provides 384-dimensional semantic representations and supports
sequences up to 512 tokens. Individual training reviews are encoded separately
rather than concatenating all reviews belonging to a business.

Before production embedding, the complete leakage-safe training corpus is
tokenised without truncation to quantify how frequently reviews exceed the
model context limit. This determines whether simple truncation or a
long-review handling strategy is required.

In [11]:
from transformers import AutoTokenizer

TEXT_MODEL_NAME = "BAAI/bge-small-en-v1.5"

text_tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_NAME
)

print("Text model:", TEXT_MODEL_NAME)
print(
    "Tokenizer model max length:",
    text_tokenizer.model_max_length
)

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Text model: BAAI/bge-small-en-v1.5
Tokenizer model max length: 512


In [12]:
TOKEN_AUDIT_BATCH_SIZE = 2048

review_token_lengths = np.empty(
    len(training_review_text),
    dtype=np.int32
)

texts = training_review_text[
    "text_clean"
].tolist()

for start in range(
    0,
    len(texts),
    TOKEN_AUDIT_BATCH_SIZE
):
    end = min(
        start + TOKEN_AUDIT_BATCH_SIZE,
        len(texts)
    )

    batch_texts = texts[start:end]

    encoded = text_tokenizer(
        batch_texts,
        add_special_tokens=True,
        truncation=False,
        padding=False
    )

    review_token_lengths[start:end] = [
        len(input_ids)
        for input_ids in encoded[
            "input_ids"
        ]
    ]


training_review_text[
    "token_count_bge"
] = review_token_lengths

print(
    "Tokenised reviews:",
    f"{len(review_token_lengths):,}"
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (579 > 512). Running this sequence through the model will result in indexing errors


Tokenised reviews: 122,233


In [13]:
token_length_summary = (
    training_review_text[
        "token_count_bge"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

token_length_summary

count    122233.000000
mean        146.010194
std         121.766423
min           3.000000
1%           22.000000
5%           31.000000
25%          64.000000
50%         111.000000
75%         189.000000
90%         293.000000
95%         379.000000
99%         604.000000
max        1257.000000
Name: token_count_bge, dtype: float64

In [14]:
token_limit_audit = pd.Series({
    "Total training reviews":
        len(training_review_text),

    "Reviews <= 128 tokens":
        (
            training_review_text[
                "token_count_bge"
            ]
            <= 128
        ).sum(),

    "Reviews > 128 tokens":
        (
            training_review_text[
                "token_count_bge"
            ]
            > 128
        ).sum(),

    "Reviews > 256 tokens":
        (
            training_review_text[
                "token_count_bge"
            ]
            > 256
        ).sum(),

    "Reviews > 384 tokens":
        (
            training_review_text[
                "token_count_bge"
            ]
            > 384
        ).sum(),

    "Reviews > 512 tokens":
        (
            training_review_text[
                "token_count_bge"
            ]
            > 512
        ).sum(),

    "Percentage > 512 tokens":
        round(
            (
                training_review_text[
                    "token_count_bge"
                ]
                > 512
            ).mean()
            * 100,
            2
        ),

    "Maximum token length":
        training_review_text[
            "token_count_bge"
        ].max()
})

token_limit_audit

Total training reviews     122233.00
Reviews <= 128 tokens       69953.00
Reviews > 128 tokens        52280.00
Reviews > 256 tokens        16733.00
Reviews > 384 tokens         5876.00
Reviews > 512 tokens         2311.00
Percentage > 512 tokens         1.89
Maximum token length         1257.00
dtype: float64

In [15]:
long_review_business_audit = (
    training_review_text
    .assign(
        exceeds_512=lambda df:
            df[
                "token_count_bge"
            ] > 512
    )
    .groupby(
        "business_id"
    )
    .agg(
        training_reviews=(
            "review_id",
            "count"
        ),

        reviews_over_512=(
            "exceeds_512",
            "sum"
        )
    )
    .reset_index()
)

long_review_business_audit[
    "percentage_over_512"
] = (
    long_review_business_audit[
        "reviews_over_512"
    ]
    /
    long_review_business_audit[
        "training_reviews"
    ]
    * 100
)


long_review_summary = pd.Series({
    "Businesses represented":
        len(long_review_business_audit),

    "Businesses with >=1 review over 512":
        (
            long_review_business_audit[
                "reviews_over_512"
            ] > 0
        ).sum(),

    "Businesses with no review over 512":
        (
            long_review_business_audit[
                "reviews_over_512"
            ] == 0
        ).sum(),

    "Maximum long reviews in one business":
        long_review_business_audit[
            "reviews_over_512"
        ].max()
})

long_review_summary

Businesses represented                  2516
Businesses with >=1 review over 512      850
Businesses with no review over 512      1666
Maximum long reviews in one business     103
dtype: int64

## Stage E2.3 — Prepare Token-Safe Review Chunks

Most training reviews fit within the 512-token context length of
BAAI/bge-small-en-v1.5. Reviews exceeding this limit are divided into
non-overlapping token chunks.

Short reviews remain unchanged. Long-review chunks are later mean-pooled back
into a single review representation before business-level aggregation. This
ensures that longer reviews do not receive greater weight solely because they
require multiple model inputs.

All original training-review text is therefore retained without introducing an
arbitrary review cap or truncating long reviews.

In [16]:
MODEL_MAX_TOKENS = text_tokenizer.model_max_length

SPECIAL_TOKENS = text_tokenizer.num_special_tokens_to_add(
    pair=False
)

MAX_CONTENT_TOKENS = (
    MODEL_MAX_TOKENS
    - SPECIAL_TOKENS
)

print("Model maximum tokens:", MODEL_MAX_TOKENS)
print("Special tokens per sequence:", SPECIAL_TOKENS)
print("Maximum content tokens:", MAX_CONTENT_TOKENS)

Model maximum tokens: 512
Special tokens per sequence: 2
Maximum content tokens: 510


In [17]:
short_reviews = (
    training_review_text[
        training_review_text[
            "token_count_bge"
        ] <= MODEL_MAX_TOKENS
    ]
    .copy()
)

long_reviews = (
    training_review_text[
        training_review_text[
            "token_count_bge"
        ] > MODEL_MAX_TOKENS
    ]
    .copy()
)

print(
    "Reviews requiring one model input:",
    f"{len(short_reviews):,}"
)

print(
    "Reviews requiring chunking:",
    f"{len(long_reviews):,}"
)

print(
    "Percentage requiring chunking:",
    f"{len(long_reviews) / len(training_review_text) * 100:.2f}%"
)

Reviews requiring one model input: 119,922
Reviews requiring chunking: 2,311
Percentage requiring chunking: 1.89%


In [18]:
short_chunk_manifest = (
    short_reviews[
        [
            "review_id",
            "user_id",
            "business_id",
            "date",
            "text_clean",
            "token_count_bge"
        ]
    ]
    .copy()
)

short_chunk_manifest[
    "chunk_number"
] = 1

short_chunk_manifest[
    "chunk_count"
] = 1

short_chunk_manifest[
    "is_chunked_review"
] = False

short_chunk_manifest[
    "chunk_text"
] = short_chunk_manifest[
    "text_clean"
]

short_chunk_manifest[
    "original_token_count"
] = short_chunk_manifest[
    "token_count_bge"
]

In [19]:
import math

long_chunk_records = []

for row in long_reviews.itertuples(
    index=False
):

    content_token_ids = text_tokenizer.encode(
        row.text_clean,
        add_special_tokens=False
    )

    chunk_count = math.ceil(
        len(content_token_ids)
        / MAX_CONTENT_TOKENS
    )

    for chunk_index in range(
        chunk_count
    ):

        start = (
            chunk_index
            * MAX_CONTENT_TOKENS
        )

        end = (
            start
            + MAX_CONTENT_TOKENS
        )

        chunk_token_ids = (
            content_token_ids[
                start:end
            ]
        )

        chunk_text = text_tokenizer.decode(
            chunk_token_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        long_chunk_records.append({
            "review_id":
                row.review_id,

            "user_id":
                row.user_id,

            "business_id":
                row.business_id,

            "date":
                row.date,

            "text_clean":
                row.text_clean,

            "token_count_bge":
                row.token_count_bge,

            "chunk_number":
                chunk_index + 1,

            "chunk_count":
                chunk_count,

            "is_chunked_review":
                True,

            "chunk_text":
                chunk_text,

            "original_token_count":
                row.token_count_bge,

            "content_tokens_in_chunk":
                len(
                    chunk_token_ids
                )
        })


long_chunk_manifest = pd.DataFrame(
    long_chunk_records
)

print(
    "Long reviews:",
    f"{len(long_reviews):,}"
)

print(
    "Generated long-review chunks:",
    f"{len(long_chunk_manifest):,}"
)

Long reviews: 2,311
Generated long-review chunks: 4,717


In [20]:
short_chunk_manifest[
    "content_tokens_in_chunk"
] = [
    len(
        text_tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )
    for text in short_chunk_manifest[
        "chunk_text"
    ]
]

In [21]:
review_chunk_manifest = (
    pd.concat(
        [
            short_chunk_manifest,
            long_chunk_manifest
        ],
        ignore_index=True
    )
    .sort_values(
        [
            "business_id",
            "review_id",
            "chunk_number"
        ]
    )
    .reset_index(drop=True)
)

review_chunk_manifest[
    "embedding_row"
] = np.arange(
    len(review_chunk_manifest)
)

print(
    "Original reviews:",
    f"{len(training_review_text):,}"
)

print(
    "Embedding chunks:",
    f"{len(review_chunk_manifest):,}"
)

print(
    "Extra model inputs caused by chunking:",
    f"{len(review_chunk_manifest) - len(training_review_text):,}"
)

Original reviews: 122,233
Embedding chunks: 124,639
Extra model inputs caused by chunking: 2,406


In [22]:
chunks_per_review = (
    review_chunk_manifest
    .groupby(
        "review_id"
    )[
        "chunk_number"
    ]
    .count()
)

chunk_structure_summary = pd.Series({
    "Original training reviews":
        len(
            training_review_text
        ),

    "Reviews represented in manifest":
        review_chunk_manifest[
            "review_id"
        ].nunique(),

    "Total embedding chunks":
        len(
            review_chunk_manifest
        ),

    "Additional chunks":
        (
            len(
                review_chunk_manifest
            )
            -
            len(
                training_review_text
            )
        ),

    "Chunked reviews":
        review_chunk_manifest.loc[
            review_chunk_manifest[
                "is_chunked_review"
            ],
            "review_id"
        ].nunique(),

    "Unchunked reviews":
        review_chunk_manifest.loc[
            ~review_chunk_manifest[
                "is_chunked_review"
            ],
            "review_id"
        ].nunique(),

    "Maximum chunks for one review":
        chunks_per_review.max(),

    "Businesses represented":
        review_chunk_manifest[
            "business_id"
        ].nunique(),

    "Maximum content tokens in any chunk":
        review_chunk_manifest[
            "content_tokens_in_chunk"
        ].max()
})

chunk_structure_summary

Original training reviews              122233
Reviews represented in manifest        122233
Total embedding chunks                 124639
Additional chunks                        2406
Chunked reviews                          2311
Unchunked reviews                      119922
Maximum chunks for one review               3
Businesses represented                   2516
Maximum content tokens in any chunk       510
dtype: int64

In [23]:
original_review_ids = set(
    training_review_text[
        "review_id"
    ]
)

manifest_review_ids = set(
    review_chunk_manifest[
        "review_id"
    ]
)


review_chunk_integrity_check = pd.Series({
    "Review ID sets exactly match":
        (
            original_review_ids
            ==
            manifest_review_ids
        ),

    "Every review has at least one chunk":
        (
            chunks_per_review
            >= 1
        ).all(),

    "No chunk exceeds safe content limit":
        (
            review_chunk_manifest[
                "content_tokens_in_chunk"
            ]
            <= MAX_CONTENT_TOKENS
        ).all(),

    "Chunk numbers start at 1":
        (
            review_chunk_manifest
            .groupby(
                "review_id"
            )[
                "chunk_number"
            ]
            .min()
            == 1
        ).all(),

    "Declared chunk count matches actual chunks":
        (
            review_chunk_manifest
            .groupby(
                "review_id"
            )
            .apply(
                lambda group:
                    len(group)
                    ==
                    group[
                        "chunk_count"
                    ].iloc[0],
                include_groups=False
            )
        ).all(),

    "All 2516 businesses retained":
        (
            review_chunk_manifest[
                "business_id"
            ].nunique()
            == 2516
        ),

    "Missing chunk text":
        review_chunk_manifest[
            "chunk_text"
        ].isna().sum(),

    "Blank chunk text":
        review_chunk_manifest[
            "chunk_text"
        ]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
})

review_chunk_integrity_check

Review ID sets exactly match                  True
Every review has at least one chunk           True
No chunk exceeds safe content limit           True
Chunk numbers start at 1                      True
Declared chunk count matches actual chunks    True
All 2516 businesses retained                  True
Missing chunk text                               0
Blank chunk text                                 0
dtype: object

In [24]:
# --------------------------------------------------
# Save frozen review-chunk manifest
# --------------------------------------------------

text_pipeline_dir = (
    processed_data_dir.parent
    / "new_orleans_text_pipeline"
)

text_pipeline_dir.mkdir(
    parents=True,
    exist_ok=True
)

review_chunk_manifest_path = (
    text_pipeline_dir
    / "new_orleans_training_review_chunk_manifest.parquet"
)

review_chunk_manifest.to_parquet(
    review_chunk_manifest_path,
    index=False,
    engine="pyarrow"
)

print(
    "Review chunk manifest saved:",
    review_chunk_manifest_path.exists()
)

print(
    "Saved rows:",
    f"{len(review_chunk_manifest):,}"
)

Review chunk manifest saved: True
Saved rows: 124,639


In [25]:
import torch
from sentence_transformers import SentenceTransformer

TEXT_MODEL_NAME = "BAAI/bge-small-en-v1.5"

if torch.cuda.is_available():
    text_device = "cuda"

elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    text_device = "mps"

else:
    text_device = "cpu"


print("Selected device:", text_device)

text_model = SentenceTransformer(
    TEXT_MODEL_NAME,
    device=text_device
)

print("Model:", TEXT_MODEL_NAME)
print(
    "Embedding dimension:",
    text_model.get_sentence_embedding_dimension()
)

print(
    "Maximum sequence length:",
    text_model.max_seq_length
)

Selected device: mps


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model: BAAI/bge-small-en-v1.5
Embedding dimension: 384
Maximum sequence length: 512


/var/folders/hl/dj9thg8x32s0k_y7xd2yfrgr0000gn/T/ipykernel_17902/191485329.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  text_model.get_sentence_embedding_dimension()


In [26]:
target_lengths = [
    32,
    64,
    128,
    256,
    384,
    450,
    500,
    510
]

smoke_rows = []

for target in target_lengths:

    nearest_index = (
        review_chunk_manifest[
            "content_tokens_in_chunk"
        ]
        .sub(target)
        .abs()
        .idxmin()
    )

    smoke_rows.append(
        review_chunk_manifest.loc[
            nearest_index
        ]
    )


smoke_test_manifest = (
    pd.DataFrame(smoke_rows)
    .drop_duplicates(
        subset="embedding_row"
    )
    .reset_index(drop=True)
)

smoke_test_manifest[
    [
        "review_id",
        "business_id",
        "chunk_number",
        "chunk_count",
        "content_tokens_in_chunk"
    ]
]

,review_id,business_id,chunk_number,chunk_count,content_tokens_in_chunk
0,cJS1ml1pAeHa-GPNKcL5og,-1XSzguS6XLN-V6MVZMg2A,1,1,32
1,RzVkuMnBnDbDcl28EBhOXQ,-0__F9fnKt8uioCKztF5Ww,1,1,64
2,d-lXgn-Dr8nG09YCJgt3NQ,-A2OLubXDsMRPNN7LqohPA,1,1,128
3,LCamW_vVHaqvigF30QG9yw,-1XSzguS6XLN-V6MVZMg2A,1,1,256
4,4UpN148Z7w_adxAP4r0L4Q,0UJqTczta018RoktahC0jw,1,1,384
5,1Wu-hf1VcYiGzcpL1X7iUg,-VlBFlHwX-Pt6Xyzs9roGw,1,1,450
6,093GJ62yo2WJqqbHdzZBQA,24f-qQokEN2UvBNHMWLm3A,1,1,500
7,RFWkTcgIpUlY2gIdMy-uvw,-0__F9fnKt8uioCKztF5Ww,1,2,510


In [27]:
smoke_embeddings = text_model.encode(
    smoke_test_manifest[
        "chunk_text"
    ].tolist(),

    batch_size=8,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Smoke embedding shape:",
    smoke_embeddings.shape
)

print(
    "Embedding dtype:",
    smoke_embeddings.dtype
)

Smoke embedding shape: (8, 384)
Embedding dtype: float32


In [28]:
smoke_norms = np.linalg.norm(
    smoke_embeddings,
    axis=1
)

bge_smoke_test_check = pd.Series({
    "Smoke-test inputs":
        len(
            smoke_test_manifest
        ),

    "Embedding rows":
        smoke_embeddings.shape[0],

    "Embedding dimension":
        smoke_embeddings.shape[1],

    "Expected dimension":
        384,

    "Row count matches":
        (
            smoke_embeddings.shape[0]
            ==
            len(
                smoke_test_manifest
            )
        ),

    "Dimension correct":
        (
            smoke_embeddings.shape[1]
            == 384
        ),

    "Contains NaN":
        np.isnan(
            smoke_embeddings
        ).any(),

    "Contains infinity":
        np.isinf(
            smoke_embeddings
        ).any(),

    "Minimum embedding norm":
        float(
            smoke_norms.min()
        ),

    "Mean embedding norm":
        float(
            smoke_norms.mean()
        ),

    "Maximum embedding norm":
        float(
            smoke_norms.max()
        ),

    "All embeddings L2-normalised":
        np.allclose(
            smoke_norms,
            1.0,
            atol=1e-5
        )
})

bge_smoke_test_check

Smoke-test inputs                   8
Embedding rows                      8
Embedding dimension               384
Expected dimension                384
Row count matches                True
Dimension correct                True
Contains NaN                    False
Contains infinity               False
Minimum embedding norm            1.0
Mean embedding norm               1.0
Maximum embedding norm            1.0
All embeddings L2-normalised     True
dtype: object

## E2.5 — Production text encoding
       (BGE encodes all 124,639 model inputs)

In [29]:
review_chunk_manifest_path = (
    text_pipeline_dir
    / "new_orleans_training_review_chunk_manifest.parquet"
)

review_chunk_manifest.to_parquet(
    review_chunk_manifest_path,
    index=False,
    engine="pyarrow"
)

print(
    "Review chunk manifest saved:",
    review_chunk_manifest_path.exists()
)

print(
    "Saved manifest rows:",
    f"{len(review_chunk_manifest):,}"
)

Review chunk manifest saved: True
Saved manifest rows: 124,639


In [30]:
bge_output_dir = (
    text_pipeline_dir
    / "bge_small_en_v1_5_embeddings"
)

bge_checkpoint_dir = (
    bge_output_dir
    / "checkpoints"
)

bge_checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "BGE output directory:",
    bge_output_dir
)

print(
    "Checkpoint directory:",
    bge_checkpoint_dir
)

BGE output directory: processed_data/new_orleans_text_pipeline/bge_small_en_v1_5_embeddings
Checkpoint directory: processed_data/new_orleans_text_pipeline/bge_small_en_v1_5_embeddings/checkpoints


In [31]:
TOTAL_CHUNKS = len(
    review_chunk_manifest
)

EMBEDDING_DIM = 384

CHECKPOINT_SIZE = 5000

if text_device == "cuda":
    BATCH_SIZE = 256

elif text_device == "mps":
    BATCH_SIZE = 128

else:
    BATCH_SIZE = 64


print("Device:", text_device)
print("Batch size:", BATCH_SIZE)
print("Checkpoint size:", CHECKPOINT_SIZE)
print("Total chunks:", f"{TOTAL_CHUNKS:,}")

Device: mps
Batch size: 128
Checkpoint size: 5000
Total chunks: 124,639


In [32]:
def get_bge_checkpoint_path(
    start_row,
    end_row
):
    return (
        bge_checkpoint_dir
        / (
            f"bge_chunks_"
            f"{start_row:06d}_"
            f"{end_row:06d}.npy"
        )
    )

In [33]:
from tqdm.auto import tqdm
import gc


checkpoint_records = []


for start_row in range(
    0,
    TOTAL_CHUNKS,
    CHECKPOINT_SIZE
):

    end_row = min(
        start_row + CHECKPOINT_SIZE,
        TOTAL_CHUNKS
    )

    checkpoint_path = (
        get_bge_checkpoint_path(
            start_row,
            end_row
        )
    )

    expected_rows = (
        end_row - start_row
    )

    # ----------------------------------------------
    # Reuse a valid checkpoint if it already exists
    # ----------------------------------------------

    if checkpoint_path.exists():

        existing_embeddings = np.load(
            checkpoint_path,
            mmap_mode="r"
        )

        checkpoint_valid = (
            existing_embeddings.shape
            ==
            (
                expected_rows,
                EMBEDDING_DIM
            )
        )

        if checkpoint_valid:

            checkpoint_records.append({
                "start_row": start_row,
                "end_row": end_row,
                "rows": expected_rows,
                "checkpoint_path":
                    str(
                        checkpoint_path
                    ),
                "status":
                    "reused"
            })

            print(
                f"Reusing checkpoint "
                f"{start_row:,}–{end_row - 1:,}"
            )

            continue

        else:

            print(
                "Invalid existing checkpoint "
                f"removed: {checkpoint_path.name}"
            )

            checkpoint_path.unlink()


    # ----------------------------------------------
    # Encode this checkpoint
    # ----------------------------------------------

    checkpoint_texts = (
        review_chunk_manifest
        .iloc[
            start_row:end_row
        ][
            "chunk_text"
        ]
        .tolist()
    )


    checkpoint_embeddings = (
        text_model.encode(
            checkpoint_texts,

            batch_size=BATCH_SIZE,

            show_progress_bar=True,

            convert_to_numpy=True,

            normalize_embeddings=True
        )
    )


    checkpoint_embeddings = (
        checkpoint_embeddings
        .astype(
            np.float32,
            copy=False
        )
    )


    # ----------------------------------------------
    # Verify before saving
    # ----------------------------------------------

    if (
        checkpoint_embeddings.shape
        !=
        (
            expected_rows,
            EMBEDDING_DIM
        )
    ):
        raise ValueError(
            "Unexpected checkpoint shape: "
            f"{checkpoint_embeddings.shape}"
        )


    if np.isnan(
        checkpoint_embeddings
    ).any():

        raise ValueError(
            "NaN detected in checkpoint."
        )


    if np.isinf(
        checkpoint_embeddings
    ).any():

        raise ValueError(
            "Infinity detected in checkpoint."
        )


    checkpoint_norms = np.linalg.norm(
        checkpoint_embeddings,
        axis=1
    )


    if not np.allclose(
        checkpoint_norms,
        1.0,
        atol=1e-5
    ):

        raise ValueError(
            "Checkpoint embeddings "
            "are not correctly normalised."
        )


    # ----------------------------------------------
    # Save checkpoint
    # ----------------------------------------------

    np.save(
        checkpoint_path,
        checkpoint_embeddings
    )


    checkpoint_records.append({
        "start_row": start_row,
        "end_row": end_row,
        "rows": expected_rows,
        "checkpoint_path":
            str(
                checkpoint_path
            ),
        "status":
            "created"
    })


    print(
        f"Saved checkpoint "
        f"{start_row:,}–{end_row - 1:,}"
    )


    del checkpoint_embeddings
    del checkpoint_texts

    gc.collect()

    if (
        text_device == "mps"
        and hasattr(
            torch.mps,
            "empty_cache"
        )
    ):
        torch.mps.empty_cache()

    elif text_device == "cuda":
        torch.cuda.empty_cache()

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Saved checkpoint 0–4,999


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Saved checkpoint 5,000–9,999


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

...

Saved checkpoint 110,000–114,999


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Saved checkpoint 115,000–119,999


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

Saved checkpoint 120,000–124,638


In [34]:
checkpoint_log = pd.DataFrame(
    checkpoint_records
)

print(
    "Checkpoint records:",
    len(checkpoint_log)
)

print(
    "Rows represented:",
    f"{checkpoint_log['rows'].sum():,}"
)

print(
    "\nCheckpoint status:"
)

print(
    checkpoint_log[
        "status"
    ].value_counts()
)

Checkpoint records: 25
Rows represented: 124,639

Checkpoint status:
status
created    25
Name: count, dtype: int64


In [35]:
all_checkpoint_embeddings = []


for start_row in range(
    0,
    TOTAL_CHUNKS,
    CHECKPOINT_SIZE
):

    end_row = min(
        start_row + CHECKPOINT_SIZE,
        TOTAL_CHUNKS
    )

    checkpoint_path = (
        get_bge_checkpoint_path(
            start_row,
            end_row
        )
    )

    if not checkpoint_path.exists():

        raise FileNotFoundError(
            f"Missing checkpoint: "
            f"{checkpoint_path.name}"
        )

    checkpoint_embeddings = np.load(
        checkpoint_path
    )

    expected_shape = (
        end_row - start_row,
        EMBEDDING_DIM
    )

    if (
        checkpoint_embeddings.shape
        != expected_shape
    ):

        raise ValueError(
            f"Bad checkpoint shape "
            f"for {checkpoint_path.name}: "
            f"{checkpoint_embeddings.shape}"
        )

    all_checkpoint_embeddings.append(
        checkpoint_embeddings
    )


bge_chunk_embeddings = np.vstack(
    all_checkpoint_embeddings
)


print(
    "Final chunk embedding matrix:",
    bge_chunk_embeddings.shape
)

Final chunk embedding matrix: (124639, 384)


In [36]:
bge_chunk_norms = np.linalg.norm(
    bge_chunk_embeddings,
    axis=1
)


bge_production_check = pd.Series({
    "Expected chunk rows":
        len(
            review_chunk_manifest
        ),

    "Embedding rows":
        bge_chunk_embeddings.shape[0],

    "Embedding dimension":
        bge_chunk_embeddings.shape[1],

    "Expected dimension":
        384,

    "Manifest and embedding rows match":
        (
            len(
                review_chunk_manifest
            )
            ==
            bge_chunk_embeddings.shape[0]
        ),

    "Embedding row order valid":
        np.array_equal(
            review_chunk_manifest[
                "embedding_row"
            ].to_numpy(),
            np.arange(
                len(
                    review_chunk_manifest
                )
            )
        ),

    "Contains NaN":
        np.isnan(
            bge_chunk_embeddings
        ).any(),

    "Contains infinity":
        np.isinf(
            bge_chunk_embeddings
        ).any(),

    "Minimum norm":
        float(
            bge_chunk_norms.min()
        ),

    "Mean norm":
        float(
            bge_chunk_norms.mean()
        ),

    "Maximum norm":
        float(
            bge_chunk_norms.max()
        ),

    "All chunk embeddings normalised":
        np.allclose(
            bge_chunk_norms,
            1.0,
            atol=1e-5
        )
})

bge_production_check

Expected chunk rows                  124639
Embedding rows                       124639
Embedding dimension                     384
Expected dimension                      384
Manifest and embedding rows match      True
Embedding row order valid              True
Contains NaN                          False
Contains infinity                     False
Minimum norm                            1.0
Mean norm                               1.0
Maximum norm                            1.0
All chunk embeddings normalised        True
dtype: object

In [37]:
bge_chunk_embedding_matrix_path = (
    bge_output_dir
    / "new_orleans_bge_training_chunk_embeddings.npy"
)

bge_chunk_embedding_index_path = (
    bge_output_dir
    / "new_orleans_bge_training_chunk_embedding_index.parquet"
)


np.save(
    bge_chunk_embedding_matrix_path,
    bge_chunk_embeddings
)

review_chunk_manifest.to_parquet(
    bge_chunk_embedding_index_path,
    index=False,
    engine="pyarrow"
)


print(
    "Chunk embedding matrix saved:",
    bge_chunk_embedding_matrix_path.exists()
)

print(
    "Chunk embedding index saved:",
    bge_chunk_embedding_index_path.exists()
)

Chunk embedding matrix saved: True
Chunk embedding index saved: True


## Stage E2.6 — Reconstruct Review-Level Text Representations

Chunk-level BGE embeddings are aggregated back to the original Yelp review
level.

For reviews represented by multiple chunks, chunk embeddings are mean-pooled
and the resulting review representation is L2-normalised. Reviews represented
by a single chunk retain the corresponding semantic representation.

This ensures that long reviews do not receive greater weight merely because
they required multiple model inputs.

In [38]:
print(
    "Chunk embedding matrix:",
    bge_chunk_embeddings.shape
)

print(
    "Chunk manifest rows:",
    len(review_chunk_manifest)
)

assert (
    len(review_chunk_manifest)
    ==
    bge_chunk_embeddings.shape[0]
)

assert (
    bge_chunk_embeddings.shape[1]
    == 384
)

assert np.array_equal(
    review_chunk_manifest[
        "embedding_row"
    ].to_numpy(),
    np.arange(
        len(review_chunk_manifest)
    )
)

print(
    "Chunk matrix/index alignment verified."
)

Chunk embedding matrix: (124639, 384)
Chunk manifest rows: 124639
Chunk matrix/index alignment verified.


In [39]:
review_codes, unique_review_ids = pd.factorize(
    review_chunk_manifest[
        "review_id"
    ],
    sort=False
)

NUM_REVIEWS = len(
    unique_review_ids
)

print(
    "Unique original reviews:",
    f"{NUM_REVIEWS:,}"
)

print(
    "Expected original reviews:",
    f"{training_review_text['review_id'].nunique():,}"
)

Unique original reviews: 122,233
Expected original reviews: 122,233


In [40]:
REVIEW_EMBEDDING_DIM = (
    bge_chunk_embeddings.shape[1]
)

review_embedding_sums = np.zeros(
    (
        NUM_REVIEWS,
        REVIEW_EMBEDDING_DIM
    ),
    dtype=np.float32
)

np.add.at(
    review_embedding_sums,
    review_codes,
    bge_chunk_embeddings
)


review_chunk_counts = np.bincount(
    review_codes
).astype(
    np.float32
)


review_embeddings = (
    review_embedding_sums
    /
    review_chunk_counts[:, None]
)


# --------------------------------------------------
# L2-normalise resulting review vectors
# --------------------------------------------------

review_embedding_norms = np.linalg.norm(
    review_embeddings,
    axis=1,
    keepdims=True
)

review_embeddings = (
    review_embeddings
    /
    np.maximum(
        review_embedding_norms,
        1e-12
    )
).astype(
    np.float32
)


print(
    "Review embedding matrix:",
    review_embeddings.shape
)

Review embedding matrix: (122233, 384)


In [41]:
review_embedding_index = (
    review_chunk_manifest[
        [
            "review_id",
            "user_id",
            "business_id",
            "date",
            "chunk_count",
            "original_token_count"
        ]
    ]
    .drop_duplicates(
        subset="review_id",
        keep="first"
    )
    .reset_index(drop=True)
)


# Factorize preserves first-occurrence order,
# so this should exactly match unique_review_ids.
assert np.array_equal(
    review_embedding_index[
        "review_id"
    ].to_numpy(),
    np.asarray(
        unique_review_ids
    )
)


review_embedding_index[
    "review_embedding_row"
] = np.arange(
    len(review_embedding_index)
)


print(
    "Review index rows:",
    len(review_embedding_index)
)

print(
    "Unique review IDs:",
    review_embedding_index[
        "review_id"
    ].nunique()
)

Review index rows: 122233
Unique review IDs: 122233


In [42]:
review_chunk_weighting_summary = pd.Series({
    "Review embeddings":
        len(
            review_embedding_index
        ),

    "Single-chunk reviews":
        (
            review_embedding_index[
                "chunk_count"
            ]
            == 1
        ).sum(),

    "Multi-chunk reviews":
        (
            review_embedding_index[
                "chunk_count"
            ]
            > 1
        ).sum(),

    "Two-chunk reviews":
        (
            review_embedding_index[
                "chunk_count"
            ]
            == 2
        ).sum(),

    "Three-chunk reviews":
        (
            review_embedding_index[
                "chunk_count"
            ]
            == 3
        ).sum(),

    "Maximum chunks per review":
        review_embedding_index[
            "chunk_count"
        ].max()
})

review_chunk_weighting_summary

Review embeddings            122233
Single-chunk reviews         119922
Multi-chunk reviews            2311
Two-chunk reviews              2216
Three-chunk reviews              95
Maximum chunks per review         3
dtype: int64

In [43]:
review_norms = np.linalg.norm(
    review_embeddings,
    axis=1
)


expected_review_ids = set(
    training_review_text[
        "review_id"
    ]
)

aggregated_review_ids = set(
    review_embedding_index[
        "review_id"
    ]
)


review_embedding_check = pd.Series({
    "Expected reviews":
        len(
            expected_review_ids
        ),

    "Review index rows":
        len(
            review_embedding_index
        ),

    "Embedding rows":
        review_embeddings.shape[0],

    "Embedding dimension":
        review_embeddings.shape[1],

    "Review IDs exactly match":
        (
            expected_review_ids
            ==
            aggregated_review_ids
        ),

    "Unique review IDs":
        review_embedding_index[
            "review_id"
        ].nunique(),

    "Duplicate review IDs":
        review_embedding_index[
            "review_id"
        ].duplicated().sum(),

    "Businesses represented":
        review_embedding_index[
            "business_id"
        ].nunique(),

    "Contains NaN":
        np.isnan(
            review_embeddings
        ).any(),

    "Contains infinity":
        np.isinf(
            review_embeddings
        ).any(),

    "Minimum norm":
        float(
            review_norms.min()
        ),

    "Mean norm":
        float(
            review_norms.mean()
        ),

    "Maximum norm":
        float(
            review_norms.max()
        ),

    "All review embeddings normalised":
        np.allclose(
            review_norms,
            1.0,
            atol=1e-5
        )
})

review_embedding_check

Expected reviews                    122233
Review index rows                   122233
Embedding rows                      122233
Embedding dimension                    384
Review IDs exactly match              True
Unique review IDs                   122233
Duplicate review IDs                     0
Businesses represented                2516
Contains NaN                         False
Contains infinity                    False
Minimum norm                           1.0
Mean norm                              1.0
Maximum norm                           1.0
All review embeddings normalised      True
dtype: object

In [44]:
bge_review_embedding_matrix_path = (
    bge_output_dir
    / "new_orleans_bge_training_review_embeddings.npy"
)

bge_review_embedding_index_path = (
    bge_output_dir
    / "new_orleans_bge_training_review_embedding_index.parquet"
)


np.save(
    bge_review_embedding_matrix_path,
    review_embeddings
)

review_embedding_index.to_parquet(
    bge_review_embedding_index_path,
    index=False,
    engine="pyarrow"
)


print(
    "Review embedding matrix saved:",
    bge_review_embedding_matrix_path.exists()
)

print(
    "Review embedding index saved:",
    bge_review_embedding_index_path.exists()
)

Review embedding matrix saved: True
Review embedding index saved: True


## Stage E2.7 — Aggregate Review Embeddings to Business Text Representations

Leakage-safe review-level BGE representations are aggregated into one textual
representation per business.

Each original training review contributes equally to its corresponding
business. Review embeddings are mean-pooled by business and the resulting
business representation is L2-normalised.

The final text matrix is explicitly aligned to the same 2,516-business order
used by the visual feature matrix. This provides a consistent business index
for subsequent non-visual and multimodal KGRec configurations.

Training-review counts are preserved as analysis metadata so that later
robustness analysis can examine the effect of textual evidence richness.

In [45]:
# --------------------------------------------------
# Canonical personalisation-business order
# --------------------------------------------------

canonical_business_index = (
    visual_feature_index
    .sort_values(
        "visual_embedding_row"
    )
    .reset_index(drop=True)
    [
        [
            "business_id",
            "visual_embedding_row"
        ]
    ]
    .copy()
)


assert np.array_equal(
    canonical_business_index[
        "visual_embedding_row"
    ].to_numpy(),
    np.arange(
        len(canonical_business_index)
    )
)


canonical_business_index[
    "business_embedding_row"
] = np.arange(
    len(canonical_business_index)
)


print(
    "Canonical businesses:",
    len(canonical_business_index)
)

print(
    "Unique business IDs:",
    canonical_business_index[
        "business_id"
    ].nunique()
)

Canonical businesses: 2516
Unique business IDs: 2516


In [46]:
business_id_to_row = dict(
    zip(
        canonical_business_index[
            "business_id"
        ],
        canonical_business_index[
            "business_embedding_row"
        ]
    )
)


review_business_rows = (
    review_embedding_index[
        "business_id"
    ]
    .map(
        business_id_to_row
    )
)


print(
    "Review embeddings:",
    len(review_embedding_index)
)

print(
    "Missing business mappings:",
    review_business_rows.isna().sum()
)

Review embeddings: 122233
Missing business mappings: 0


In [47]:
review_business_rows = (
    review_business_rows
    .astype(np.int32)
    .to_numpy()
)

In [48]:
NUM_BUSINESSES = len(
    canonical_business_index
)

TEXT_EMBEDDING_DIM = (
    review_embeddings.shape[1]
)


business_text_sums = np.zeros(
    (
        NUM_BUSINESSES,
        TEXT_EMBEDDING_DIM
    ),
    dtype=np.float32
)


np.add.at(
    business_text_sums,
    review_business_rows,
    review_embeddings
)


business_review_counts = np.bincount(
    review_business_rows,
    minlength=NUM_BUSINESSES
).astype(
    np.float32
)


if np.any(
    business_review_counts == 0
):
    raise ValueError(
        "At least one personalisation business "
        "has no training review embeddings."
    )


business_text_embeddings = (
    business_text_sums
    /
    business_review_counts[:, None]
)

In [49]:
business_text_norms = np.linalg.norm(
    business_text_embeddings,
    axis=1,
    keepdims=True
)


business_text_embeddings = (
    business_text_embeddings
    /
    np.maximum(
        business_text_norms,
        1e-12
    )
).astype(
    np.float32
)


print(
    "Business text embedding matrix:",
    business_text_embeddings.shape
)

Business text embedding matrix: (2516, 384)


In [50]:
business_text_embedding_index = (
    canonical_business_index
    .copy()
)


business_text_embedding_index[
    "text_embedding_row"
] = np.arange(
    len(
        business_text_embedding_index
    )
)


business_text_embedding_index[
    "training_review_count"
] = (
    business_review_counts
    .astype(int)
)


business_text_embedding_index[
    "text_model"
] = TEXT_MODEL_NAME


business_text_embedding_index[
    "aggregation_method"
] = (
    "mean_review_embeddings_then_l2_normalise"
)

In [51]:
text_analysis_metadata = (
    business_text_summary[
        [
            "business_id",
            "training_reviewer_count",
            "total_words",
            "mean_review_words",
            "median_review_words",
            "maximum_review_words"
        ]
    ]
    .copy()
)


business_text_embedding_index = (
    business_text_embedding_index
    .merge(
        text_analysis_metadata,
        on="business_id",
        how="left",
        validate="one_to_one"
    )
)

In [52]:
final_business_text_summary = pd.Series({
    "Businesses":
        len(
            business_text_embedding_index
        ),

    "Total source reviews":
        business_text_embedding_index[
            "training_review_count"
        ].sum(),

    "Minimum reviews per business":
        business_text_embedding_index[
            "training_review_count"
        ].min(),

    "Median reviews per business":
        business_text_embedding_index[
            "training_review_count"
        ].median(),

    "Maximum reviews per business":
        business_text_embedding_index[
            "training_review_count"
        ].max(),

    "Businesses with zero reviews":
        (
            business_text_embedding_index[
                "training_review_count"
            ]
            == 0
        ).sum()
})

final_business_text_summary

Businesses                        2516.0
Total source reviews            122233.0
Minimum reviews per business         1.0
Median reviews per business         19.0
Maximum reviews per business      1271.0
Businesses with zero reviews         0.0
dtype: float64

In [53]:
final_text_norms = np.linalg.norm(
    business_text_embeddings,
    axis=1
)


expected_business_ids = set(
    personalisation_businesses[
        "business_id"
    ]
)

text_business_ids = set(
    business_text_embedding_index[
        "business_id"
    ]
)


business_text_embedding_check = pd.Series({
    "Expected businesses":
        len(
            expected_business_ids
        ),

    "Business index rows":
        len(
            business_text_embedding_index
        ),

    "Embedding rows":
        business_text_embeddings.shape[0],

    "Embedding dimension":
        business_text_embeddings.shape[1],

    "Expected dimension":
        384,

    "Business IDs exactly match":
        (
            expected_business_ids
            ==
            text_business_ids
        ),

    "Unique business IDs":
        business_text_embedding_index[
            "business_id"
        ].nunique(),

    "Duplicate business IDs":
        business_text_embedding_index[
            "business_id"
        ].duplicated().sum(),

    "Total source reviews":
        business_text_embedding_index[
            "training_review_count"
        ].sum(),

    "Expected source reviews":
        len(
            training_review_text
        ),

    "All businesses have training text":
        (
            business_text_embedding_index[
                "training_review_count"
            ] > 0
        ).all(),

    "Text/visual business order identical":
        np.array_equal(
            business_text_embedding_index[
                "business_id"
            ].to_numpy(),
            canonical_business_index[
                "business_id"
            ].to_numpy()
        ),

    "Contains NaN":
        np.isnan(
            business_text_embeddings
        ).any(),

    "Contains infinity":
        np.isinf(
            business_text_embeddings
        ).any(),

    "Minimum norm":
        float(
            final_text_norms.min()
        ),

    "Mean norm":
        float(
            final_text_norms.mean()
        ),

    "Maximum norm":
        float(
            final_text_norms.max()
        ),

    "All business text embeddings normalised":
        np.allclose(
            final_text_norms,
            1.0,
            atol=1e-5
        )
})

business_text_embedding_check

Expected businesses                          2516
Business index rows                          2516
Embedding rows                               2516
Embedding dimension                           384
Expected dimension                            384
Business IDs exactly match                   True
Unique business IDs                          2516
Duplicate business IDs                          0
Total source reviews                       122233
Expected source reviews                    122233
All businesses have training text            True
Text/visual business order identical         True
Contains NaN                                False
Contains infinity                           False
Minimum norm                                  1.0
Mean norm                                     1.0
Maximum norm                                  1.0
All business text embeddings normalised      True
dtype: object

In [54]:
bge_business_embedding_matrix_path = (
    bge_output_dir
    / "new_orleans_bge_training_business_embeddings.npy"
)

bge_business_embedding_index_path = (
    bge_output_dir
    / "new_orleans_bge_training_business_embedding_index.parquet"
)


np.save(
    bge_business_embedding_matrix_path,
    business_text_embeddings
)


business_text_embedding_index.to_parquet(
    bge_business_embedding_index_path,
    index=False,
    engine="pyarrow"
)


print(
    "Business text embedding matrix saved:",
    bge_business_embedding_matrix_path.exists()
)

print(
    "Business text embedding index saved:",
    bge_business_embedding_index_path.exists()
)

Business text embedding matrix saved: True
Business text embedding index saved: True
